# Training YOLO11n ourselves: from scratch (B) and fine-tuned (C)

Kontur suggests objects with **YOLO11n pretrained on COCO** by Ultralytics (model **A**).
This notebook trains the **same architecture ourselves**, two ways:

| | Model | Weights at the start |
|---|---|---|
| **B** | **YOLO11n from scratch** | **random** (built from the architecture file `yolo11n.yaml`) |
| **C** | YOLO11n fine-tuned | Kontur's own pretrained `weights/yolo11n.pt` |

Same data, epochs, image size, seed and settings for both; the **only** difference is
the starting point. The comparison with A is in **`eval/04_scratch_vs_pretrained.ipynb`**;
this notebook only builds the data and creates the models.

**Task:** detect the five classes of Kontur's effort study: person, car, bicycle, dog, chair.

**Data:** COCO 2017 [Lin et al. 2014], downloaded by this notebook:

| Split | From | Used for |
|---|---|---|
| `train` | a class-balanced subset of COCO **train**2017 | training B and C |
| `val` | other COCO **train**2017 images, never trained on | picking the best epoch |
| `test` | every COCO **val**2017 image with these classes | **only** the final comparison (notebook `eval/04`) |

**How to run:** check *Settings*, then Run All. Downloads skip files already there, so
re-running is safe. If the data lab stops a long run, set `RESUME = True` and run the
training cell again.

**Outputs** (committed as evidence): `training/results/` (environment, dataset, each run's
curves and settings) and `training/trained/` (the weights of B and C).

## Settings

In [ ]:
from pathlib import Path
import os

HERE = Path.cwd()                      # the training/ folder (Jupyter runs a notebook from its own folder)
if not (HERE / "train_yolo11n_coco5.ipynb").exists() and (HERE / "training").is_dir():
    HERE = HERE / "training"           # started from the repository root instead
ROOT = HERE.parent                     # the repository root

# Where COCO goes. Default: training/data. On the data lab you can point it anywhere,
# e.g. DATA_DIR = Path("/scratch/<you>/coco"), or set the environment variable KONTUR_DATA.
# eval/04 reads the same place, so set KONTUR_DATA (or edit both notebooks) if you change it.
DATA_DIR = Path(os.environ.get("KONTUR_DATA", HERE / "data")).resolve()

CLASSES = ["person", "car", "bicycle", "dog", "chair"]   # COCO names; the study classes
MAX_TRAIN_IMAGES = 20_000   # class-balanced cap for the train split
VAL_IMAGES = 1_000          # held-out train2017 images, for picking the best epoch
EPOCHS = 100                # the same for B and C, so they are comparable
PATIENCE = EPOCHS           # no early stopping: B and C both get every epoch
IMGSZ = 640
BATCH = -1                  # -1 = Ultralytics AutoBatch (fits the GPU's memory)
WORKERS = 8                 # data-loading processes
CACHE = False               # "ram" speeds training up if the data lab has ~25 GB RAM to spare
DEVICE = 0                  # first GPU; "cpu" only for a quick test
SEED = 2026                 # same seed as Kontur's evaluation
DOWNLOAD_THREADS = 32

RUN_B = True                # train from scratch
RUN_C = True                # fine-tune from the pretrained weights
RESUME = False              # True: continue an interrupted run from its last.pt

RUNS = HERE / "runs"                    # Ultralytics logs (not committed)
RESULTS = HERE / "results"              # evidence (committed)
TRAINED = HERE / "trained"              # our trained weights (committed)
PRETRAINED = ROOT / "weights" / "yolo11n.pt"   # the exact file Kontur uses; downloaded there if missing
for d in (DATA_DIR, RUNS, RESULTS, TRAINED, PRETRAINED.parent):
    d.mkdir(parents=True, exist_ok=True)
print("notebook folder:", HERE)
print("data folder:    ", DATA_DIR)

## Environment (recorded as evidence)

In [ ]:
import json, platform, sys, time
import torch, ultralytics

env = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else None,
    "started": time.strftime("%Y-%m-%d %H:%M:%S"),
    "settings": dict(CLASSES=CLASSES, MAX_TRAIN_IMAGES=MAX_TRAIN_IMAGES, VAL_IMAGES=VAL_IMAGES, EPOCHS=EPOCHS,
                     PATIENCE=PATIENCE, IMGSZ=IMGSZ, BATCH=BATCH, SEED=SEED, DEVICE=DEVICE),
}
(RESULTS / "environment.json").write_text(json.dumps(env, indent=1))
print(json.dumps(env, indent=1))
if not env["cuda_available"]:
    print("\nWARNING: no GPU found. Training will be extremely slow; check the data lab session has a GPU.")

## Step 1: download the COCO annotations (241 MB, once)

From the official COCO site. Only the two instance files are unpacked.

In [ ]:
import shutil, time, urllib.request
from pathlib import Path

UA = {"User-Agent": "Mozilla/5.0 (coco-download)"}

def fetch(url, dest, tries=4, timeout=60):
    """Download url to dest unless it is already there. Retries, then explains."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    err = None
    for k in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout) as r, \
                 open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            tmp.replace(dest)
            return dest
        except Exception as e:
            err = e
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"could not download {url}: {err}")

import zipfile

ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANN = DATA_DIR / "annotations"
NEED = ["instances_train2017.json", "instances_val2017.json"]
if not all((ANN / n).exists() for n in NEED):
    z = DATA_DIR / "annotations_trainval2017.zip"
    print("downloading", ANN_URL, "(241 MB) ...")
    try:
        fetch(ANN_URL, z, timeout=600)
    except RuntimeError as e:
        raise RuntimeError(f"{e}\nIf the data lab blocks this, download the zip in a browser and put it at {z}") from None
    with zipfile.ZipFile(z) as zf:
        for n in NEED:
            zf.extract(f"annotations/{n}", DATA_DIR)
print("annotations ready:", sorted(p.name for p in ANN.glob("*.json")))

## Step 2: choose the images

- **Only non-crowd objects of the five classes** count; other COCO objects in the photos are simply not labelled.
- **Train** is class-balanced: rarest class first, each class gets an equal share of `MAX_TRAIN_IMAGES`, and leftovers fill the rest (COCO is dominated by people).
- **Val** is `VAL_IMAGES` other train2017 images, never trained on.
- **Test** is every val2017 image that contains at least one of the classes. It is written down here and used only by `eval/04`.

In [ ]:
import random
from pycocotools.coco import COCO

coco_tr = COCO(str(ANN / "instances_train2017.json"))
coco_te = COCO(str(ANN / "instances_val2017.json"))
cat_ids = {n: coco_tr.getCatIds(catNms=[n])[0] for n in CLASSES}   # COCO category id per class
assert len(cat_ids) == len(CLASSES), cat_ids

def images_with(coco, cid):
    return sorted({a["image_id"] for a in coco.loadAnns(coco.getAnnIds(catIds=[cid], iscrowd=False))})

rng = random.Random(SEED)
per_class = {n: images_with(coco_tr, c) for n, c in cat_ids.items()}
quota = MAX_TRAIN_IMAGES // len(CLASSES)
train_ids = set()
for n in sorted(CLASSES, key=lambda n: len(per_class[n])):        # rarest class first
    pool = [i for i in per_class[n] if i not in train_ids]
    rng.shuffle(pool)
    train_ids.update(pool[:quota])
rest = sorted(set().union(*per_class.values()) - train_ids)
rng.shuffle(rest)
train_ids.update(rest[:max(0, MAX_TRAIN_IMAGES - len(train_ids))])
rest = [i for i in rest if i not in train_ids]
val_ids = rest[:VAL_IMAGES]
test_ids = sorted(set().union(*(images_with(coco_te, c) for c in cat_ids.values())))
split = {"train": sorted(train_ids), "val": sorted(val_ids), "test": test_ids}
assert not (set(split["train"]) & set(split["val"])), "train and val overlap"

def count(coco, ids):
    ids = set(ids)
    return {n: sum(1 for a in coco.loadAnns(coco.getAnnIds(catIds=[c], iscrowd=False)) if a["image_id"] in ids)
            for n, c in cat_ids.items()}

counts = {"train": count(coco_tr, split["train"]), "val": count(coco_tr, split["val"]), "test": count(coco_te, split["test"])}
(RESULTS / "dataset.json").write_text(json.dumps(
    {"classes": CLASSES, "coco_category_ids": cat_ids, "seed": SEED,
     "images": {k: len(v) for k, v in split.items()}, "objects_per_class": counts,
     "image_ids": split}, indent=1))
for k, v in split.items():
    print(f"{k:5s} {len(v):6d} images   objects: {counts[k]}")

## Step 3: download the images (about 3–4 GB, parallel, skips what is already there)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

IMG = DATA_DIR / "images"
LBL = DATA_DIR / "labels"
src = {"train": coco_tr, "val": coco_tr, "test": coco_te}

failed = []
for name, ids in split.items():
    coco = src[name]
    jobs = [(coco.imgs[i]["coco_url"], IMG / name / coco.imgs[i]["file_name"]) for i in ids]
    todo = [j for j in jobs if not (j[1].exists() and j[1].stat().st_size > 0)]
    with ThreadPoolExecutor(DOWNLOAD_THREADS) as pool:
        futures = {pool.submit(fetch, url, dest): url for url, dest in todo}
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"{name} images"):
            if f.exception() is not None:
                failed.append(futures[f])
    print(f"{name}: {len(jobs) - len(todo)} already here, {len(todo)} downloaded")
if failed:
    print(f"\n{len(failed)} images failed, e.g. {failed[:3]}. Run this cell again; it only fetches what is missing.")

## Step 4: write YOLO labels and the dataset file

One `.txt` per image: `class cx cy w h`, normalised to 0–1. Ultralytics finds an image's
labels by replacing `images/` with `labels/` in its path.

In [ ]:
idx = {c: k for k, c in enumerate(cat_ids.values())}   # COCO category id -> 0..4

def write_labels(name, coco, ids):
    out = LBL / name
    out.mkdir(parents=True, exist_ok=True)
    n_obj = 0
    for i in ids:
        im = coco.imgs[i]
        W, H = im["width"], im["height"]
        lines = []
        for a in coco.loadAnns(coco.getAnnIds(imgIds=[i], catIds=list(cat_ids.values()), iscrowd=False)):
            x, y, w, h = a["bbox"]
            x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            lines.append(f"{idx[a['category_id']]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
                         f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        (out / (Path(im["file_name"]).stem + ".txt")).write_text("\n".join(lines) + ("\n" if lines else ""))
        n_obj += len(lines)
    return n_obj

for name, ids in split.items():
    print(f"{name}: {write_labels(name, src[name], ids)} objects labelled")

DATA_YAML = DATA_DIR / "coco5.yaml"
DATA_YAML.write_text(
    f"path: {DATA_DIR.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
    + "".join(f"  {k}: {n}\n" for k, n in enumerate(CLASSES)))
print(DATA_YAML.read_text())

## Step 5: the training function (shared by B and C)

Both runs use identical settings; only the starting weights differ. When a run ends, its
best weights (chosen on the **val** split) are copied to `trained/`, and its curves and
settings to `results/`.

In [ ]:
from ultralytics import YOLO

hours = {}

def train(tag, start, pretrained):
    last = RUNS / tag / "weights" / "last.pt"
    t0 = time.perf_counter()
    if RESUME and last.exists():
        YOLO(str(last)).train(resume=True)
    else:
        YOLO(start).train(
            data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
            workers=WORKERS, cache=CACHE, device=DEVICE, seed=SEED, deterministic=True,
            pretrained=pretrained, project=str(RUNS), name=tag, exist_ok=True, plots=True)
    hours[tag] = round((time.perf_counter() - t0) / 3600, 2)
    best = RUNS / tag / "weights" / "best.pt"
    dest = TRAINED / f"yolo11n_{tag}_coco5.pt"
    shutil.copy(best, dest)
    for f in ("results.csv", "results.png", "confusion_matrix.png", "args.yaml"):
        if (RUNS / tag / f).exists():
            shutil.copy(RUNS / tag / f, RESULTS / f"{tag}_{f}")
    env["training_hours"] = hours
    (RESULTS / "environment.json").write_text(json.dumps(env, indent=1))
    print(f"{tag}: {hours[tag]} h, best weights -> {dest}")
    return dest

## B: YOLO11n from scratch

`YOLO("yolo11n.yaml")` builds the network from its architecture file: **every weight starts
random**. `pretrained=False` makes sure nothing is copied in. (Ultralytics may download a
small model once for its mixed-precision self-test; that model is not used for training.)

In [ ]:
if RUN_B:
    weights_B = train("B_scratch", "yolo11n.yaml", pretrained=False)

## C: YOLO11n fine-tuned from Kontur's pretrained weights

Same data, epochs and settings as B. Ultralytics copies every pretrained layer and
re-creates only the final layer, for 5 classes instead of 80.

In [ ]:
if RUN_C:
    weights_C = train("C_finetune", str(PRETRAINED), pretrained=True)

## Done

- `trained/yolo11n_B_scratch_coco5.pt` and `trained/yolo11n_C_finetune_coco5.pt`: our models.
- `results/`: environment and hours, the dataset (image ids of every split), each run's
  `results.csv`, `results.png`, `confusion_matrix.png` and `args.yaml`.

**Next: run `eval/04_scratch_vs_pretrained.ipynb`** to compare A, B and C on the test split.